# Camada Silver — `financeiro`

**Disciplina:** Mineração de Dados — FATEC Matão-SP (Prof. Rodrigo Vidrich)  
**Projeto:** Trabalho Final (P2) — Arquitetura Medallion (`Bronze → Silver → Gold → Power BI`)  
**Integrantes:** Jerônimo Barbieri Junior, Luan Chierato Oliveira, Miguel Miranda Morandini, Victor Hugo Malipense Testi  
**Data:** 13/11/2026  

**Técnicas aplicadas:** Remoção de registros sem chave `id_aluno`, Padronização Numérica/Datas, Imputação por Mediana (`valor_mensalidade`) e Moda (`situacao_financeira`), Tratamento contextual (`forma_pagamento`), Validação (`>= 0`) e Outliers (IQR e Z-Score).

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (candidate / "notebooks" / "utils").exists():
        if str(candidate / "notebooks") not in sys.path:
            sys.path.insert(0, str(candidate / "notebooks"))
        break

from utils.silver_utils import (
    anonymize_name_lgpd,
    detect_outliers_iqr,
    detect_outliers_zscore,
    fix_mojibake_text,
    get_silver_paths,
    load_bronze,
    normalize_id_aluno,
    parse_br_float,
    parse_date_col,
    save_silver_outputs,
)

ROOT_DIR, BRONZE_DIR, SILVER_DIR = get_silver_paths()

In [ ]:
df = load_bronze("financeiro", BRONZE_DIR)
rows_in = len(df)

df["id_lancamento"] = normalize_id_aluno(df["id_lancamento"])
df["id_aluno"] = normalize_id_aluno(df["id_aluno"])

# Remove registros sem id_aluno (impossível vincular ao aluno/matrícula)
sem_id = int(df["id_aluno"].isna().sum())
df = df.dropna(subset=["id_aluno"]).copy()

df["competencia"] = df["competencia"].astype(str).str.strip().replace("", "Não Informada")
df["data_pagamento"] = parse_date_col(df["data_pagamento"])

for col in ["valor_mensalidade", "valor_desconto", "valor_pago"]:
    df[col] = parse_br_float(df[col]).clip(lower=0)

# Imputação por Mediana em valor_mensalidade ausente
mens_na = int(df["valor_mensalidade"].isna().sum())
mens_mediana = float(df["valor_mensalidade"].median())
df["valor_mensalidade"] = df["valor_mensalidade"].fillna(mens_mediana)
df["valor_desconto"] = df["valor_desconto"].fillna(0.0)
df["valor_pago"] = df["valor_pago"].fillna(0.0)

# Padronização e imputação pela Moda em situacao_financeira
df["situacao_financeira"] = df["situacao_financeira"].astype(str).str.strip().str.upper().replace("", pd.NA)
moda_sit = str(df["situacao_financeira"].mode(dropna=True).iloc[0])
sit_na = int(df["situacao_financeira"].isna().sum())
df["situacao_financeira"] = df["situacao_financeira"].fillna(moda_sit).astype("category")

# Forma de pagamento vazia (esperada para títulos não pagos) -> categoria explícita
df["forma_pagamento"] = (
    df["forma_pagamento"].astype(str).str.strip().replace("", "NÃO PAGO / N/A").str.upper().astype("category")
)

df["flag_mensalidade_outlier_iqr"], iqr_mens = detect_outliers_iqr(df["valor_mensalidade"])
df["flag_mensalidade_outlier_zscore"], z_mens = detect_outliers_zscore(df["valor_mensalidade"])

dups_removed = int(df.duplicated(subset=["id_lancamento"]).sum())
df = df.drop_duplicates(subset=["id_lancamento"]).reset_index(drop=True)

log = {
    "dataset": "financeiro",
    "rows_in": rows_in,
    "rows_removed_null_id_aluno": sem_id,
    "rows_out": int(len(df)),
    "duplicates_removed": dups_removed,
    "valor_mensalidade_imputado_mediana": mens_na,
    "valor_mensalidade_mediana": round(mens_mediana, 2),
    "situacao_imputada_moda": sit_na,
    "situacao_moda": moda_sit,
    "mensalidade_outliers_iqr": iqr_mens,
    "mensalidade_outliers_zscore": z_mens,
}
save_silver_outputs(df, log, "financeiro", SILVER_DIR)
print(json.dumps(log, ensure_ascii=False, indent=2))
df.head(5)